In [1]:
!pip install -q -U google-generativeai

In [4]:
import time
import json

import pandas as pd

import google.generativeai as genai
from google.colab import userdata
from google.api_core import exceptions

In [3]:
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

In [ ]:
genai_model = genai.GenerativeModel('gemini-2.5-flash-lite')

#### Load the data from a json file.

In [ ]:
with open('Data/data.json', 'rt') as f_in:
    documents = json.load(f_in)

#### Template for all the prompts to generate questions for data entries.

In [ ]:
prompt_templet = """

    You're a friendly and knowledgeable filmmaking tutor! Your task is to transform a single filmmaking concept into 5 engaging, learner-friendly questions. These questions will help a student truly understand and apply the topic.

    Here is the concept you'll be working with, pulled from our filmmaking knowledge base:

    Type: {type}
    Term: {term}
    Definition: {definition}
    Extra: {extra}

    When generating your questions, keep these guidelines in mind:

    - Make sure you include a variety. At least one question should be a simple recall question (e.g., "what is..."), and at least one should test practical application (e.g., "when would you use...").
    - Keep the questions short and easy to understand.
    - Only provide the questions themselves, not the answers.
    - The output should be a JSON array of strings.
    - Do not use code blocks.

""".strip()

#### Generating Questions.

In [ ]:
done = 0
MAX_RETRIES = 5
DELAY_SECONDS = 5
response = {}

In [ ]:
for record in documents:

    for attempt in range(MAX_RETRIES):
        try:
            prompt = prompt_templet.format(**record)
            questions = genai_model.generate_content(prompt)
            questions[record['id']] = questions.text

            break

        except exceptions.InternalServerError as e:
            print(f"InternalServerError: (Attempt {attempt + 1}/{MAX_RETRIES})")
            time.sleep(DELAY_SECONDS * (2 ** attempt))

        except exceptions.ResourceExhausted as e:

            print(f"ResourceExhausted: {e}. Rate limit reached. Waiting for 15 seconds...")
            time.sleep(15)
            continue

        except Exception as e:
            print(f"An unexpected error occurred. Skipping... Error: {e}")
            break

    else:
        print(f"Failed to process record ID: {record['id']} after {MAX_RETRIES} attempts.")

    done += 1
    if done % 15 == 0:
        time.sleep(30)
        print(f'Records completed: {done}/{len(documents)}')

print("Processing complete.")

In [ ]:
if len(response) != len(documents):

    for record in documents:

        if record['id'] not in response:
            prompt = prompt_templet.format(**record)
            questions = genai_model.generate_content(prompt)
            response[record'id'] = questions.text

print(f'reponse entries: {len(response)}')
print(f'  total records: {len(documents)}')

In [ ]:
data = []

for id, questions in response.items():
    for question in json.loads(questions):
        row = {'question': question, 'id': id}
        data.append(row)

In [ ]:
groud_truth = pd.DataFrame(data)

In [ ]:
pd.to_csv(
    'ground_truth.csv',
    index = False
)